# Entrenament model reconeixement de cotxes

**Dataset:** Stanford Cars (16.185 imatges, 196 classes marca+model+any)  
**Model:** EfficientNet-B3  
**GPU:** T4 gratuita de Colab  
**Temps:** ~1-2 hores

> **IMPORTANT:** Ves a `Runtime → Change runtime type → T4 GPU` abans d'executar

In [ ]:
# Instal·lar dependències
!pip install -q datasets timm torchvision Pillow tqdm

In [ ]:
# Verificar GPU
import torch
print(f'GPU: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Model GPU: {torch.cuda.get_device_name(0)}')
else:
    print('ATENCIO: Sense GPU. Ves a Runtime > Change runtime type > T4 GPU')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# Descarregar Stanford Cars Dataset
from datasets import load_dataset
print('Descarregant Stanford Cars (~1.8GB)...')
dataset = load_dataset('tanganke/stanford_cars')
class_names = dataset['train'].features['label'].names
num_classes = len(class_names)
print(f'Train: {len(dataset["train"])} | Test: {len(dataset["test"])} | Classes: {num_classes}')
print('Exemples:', class_names[:5])

In [ ]:
# DataLoaders
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader

IMG_SIZE = 300
BATCH_SIZE = 32

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])
val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

class CarsDS(Dataset):
    def __init__(self, ds, tf):
        self.ds, self.tf = ds, tf
    def __len__(self):
        return len(self.ds)
    def __getitem__(self, i):
        x = self.ds[i]
        return self.tf(x['image'].convert('RGB')), x['label']

train_loader = DataLoader(CarsDS(dataset['train'], train_tf), batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(CarsDS(dataset['test'],  val_tf),   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print(f'Batches train: {len(train_loader)} | val: {len(val_loader)}')

In [ ]:
# Carregar model EfficientNet-B3
import timm
import torch.nn as nn
model = timm.create_model('efficientnet_b3', pretrained=True, num_classes=num_classes)
model = model.to(device)
print(f'Parametres: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

In [ ]:
# Entrenament
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm import tqdm

EPOCHS = 20
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_acc = 0.0
for epoch in range(EPOCHS):
    model.train()
    loss_sum = 0
    for imgs, lbls in tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS}'):
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        loss = criterion(model(imgs), lbls)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item()
    model.eval()
    ok = total = 0
    with torch.no_grad():
        for imgs, lbls in val_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            _, pred = model(imgs).max(1)
            total += lbls.size(0)
            ok += pred.eq(lbls).sum().item()
    acc = 100. * ok / total
    scheduler.step()
    print(f'Epoch {epoch+1:02}/{EPOCHS} | Loss: {loss_sum/len(train_loader):.4f} | Acc: {acc:.2f}%')
    if acc > best_acc:
        best_acc = acc
        torch.save({'state_dict': model.state_dict(), 'class_names': class_names, 'num_classes': num_classes}, 'model_cotxes.pth')
        print(f'  NOU MILLOR MODEL: {acc:.2f}%')
print(f'\nMillor precisio: {best_acc:.2f}%')

In [ ]:
# Prova amb una imatge
import requests
from PIL import Image
from io import BytesIO

def identificar(url, top=3):
    img = Image.open(BytesIO(requests.get(url).content)).convert('RGB')
    t = val_tf(img).unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad():
        probs = torch.softmax(model(t), dim=1)
        vals, idxs = probs.topk(top)
    for p, i in zip(vals[0], idxs[0]):
        print(f'  {class_names[i]:55s} {p.item()*100:.1f}%')

print('Prova amb foto de buscocotxe.ad:')
identificar('https://www.buscocotxe.ad/uploads/fotos_items/589619/1775942632_img_7378.jpeg')

In [ ]:
# Descarregar model
from google.colab import files
files.download('model_cotxes.pth')
print('Model descarregat!')